In [1]:
import os
import glob
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error


def process_single_well_delta(filepath, is_test=False):
 
    well_name = os.path.basename(filepath).split('__')[0]
    dir_path = os.path.dirname(filepath)
    df = pd.read_csv(filepath)
    df['WELLNAME'] = well_name
    
    df['C_input'] = df['TVT_input'] + df['Z']
    df['Interpolated_C'] = df['C_input'].interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(20742.0)
    
    if not is_test and 'TVT' in df.columns:
        df['Target_C'] = df['TVT'] + df['Z']
        df['Target_Residual'] = df['Target_C'] - df['Interpolated_C']
        
        # Target: First difference of the Residual (The Delta)
        df['Target_Delta_Residual'] = df['Target_Residual'].diff().fillna(0.0)
    
    dz = df['Z'].diff().fillna(0.0)
    dmd = df['MD'].diff().fillna(1.0)  # Avoid division by zero
    df['dZ_dMD'] = (dz / (dmd + 1e-8)).fillna(0.0)
    df['dZ'] = dz
    df['dMD'] = dmd
    
    dx = df['X'].diff().fillna(0.0)
    dy = df['Y'].diff().fillna(0.0)
    df['dX'] = dx
    df['dY'] = dy
    df['azimuth'] = np.arctan2(dy, dx + 1e-8)
    
    df['GR_clean'] = df['GR'].interpolate(limit_direction='both').ffill().bfill().fillna(85.0)
    df['GR_roll_10'] = df['GR_clean'].rolling(window=10, center=True, min_periods=1).mean()
    df['GR_roll_50'] = df['GR_clean'].rolling(window=50, center=True, min_periods=1).mean()
    df['GR_roll_250'] = df['GR_clean'].rolling(window=250, center=True, min_periods=1).mean()
    
    df['dGR'] = df['GR_clean'].diff().fillna(0.0)
    df['dGR_roll_10'] = df['GR_roll_10'].diff().fillna(0.0)
    df['dGR_roll_50'] = df['GR_roll_50'].diff().fillna(0.0)
    
    known_c_vals = df['C_input'].dropna()
    df['well_known_mean_C'] = known_c_vals.mean() if len(known_c_vals) > 0 else 20742.0
    df['well_known_std_C'] = known_c_vals.std() if len(known_c_vals) > 1 else 35.0
    
    df['relative_md'] = df['MD'] - df['MD'].min()
    df['is_eval_zone'] = df['TVT_input'].isna().astype(int)
    
    if 'row_index' not in df.columns:
        df['row_index'] = np.arange(len(df))
        
    typewell_filename = f"{well_name}__typewell.csv"
    typewell_path = os.path.join(dir_path, typewell_filename)
    
    df['Typewell_GR_at_Est_TVT'] = 85.0
    df['GR_to_Typewell_Ratio'] = 1.0
    df['GR_to_Typewell_Diff'] = 0.0
    df['typewell_alignment_score'] = 0.0
    
    if os.path.exists(typewell_path):
        try:
            typewell_df = pd.read_csv(typewell_path).sort_values('TVT')
            est_tvt = df['Interpolated_C'] - df['Z']
            
            typewell_gr_at_est = np.interp(est_tvt, typewell_df['TVT'], typewell_df['GR'])
            df['Typewell_GR_at_Est_TVT'] = typewell_gr_at_est
            df['GR_to_Typewell_Ratio'] = df['GR_clean'] / (typewell_gr_at_est + 1e-5)
            df['GR_to_Typewell_Diff'] = df['GR_clean'] - typewell_gr_at_est
            
            known_mask = df['is_eval_zone'] == 0
            if known_mask.sum() > 5:
                known_tvt = df.loc[known_mask, 'TVT_input']
                known_horiz_gr = df.loc[known_mask, 'GR_clean']
                known_typewell_gr = np.interp(known_tvt, typewell_df['TVT'], typewell_df['GR'])
                
                # Check variance to prevent divide-by-zero or NaN correlations
                if known_horiz_gr.std() > 1e-5 and pd.Series(known_typewell_gr).std() > 1e-5:
                    corr = pd.Series(known_horiz_gr).corr(pd.Series(known_typewell_gr))
                    if not np.isnan(corr):
                        df['typewell_alignment_score'] = corr
        except Exception:
            pass  # Gracefully fallback if file parsing fails
            
    return df

def load_all_delta_wells(directory, is_test=False):
    files = glob.glob(os.path.join(directory, "*__horizontal_well.csv"))
    if not files:
        return pd.DataFrame()
    processed_dfs = [process_single_well_delta(f, is_test=is_test) for f in files]
    return pd.concat(processed_dfs, ignore_index=True)


def reconstruct_well_using_bridge(well_df, pred_deltas):
   
    well_df = well_df.copy().reset_index(drop=True)
    n = len(well_df)
    
    pred_residuals = np.zeros(n)
    is_eval = well_df['is_eval_zone'].values
    segments = []
    in_segment = False
    start = None
    
    for i in range(n):
        if is_eval[i] and not in_segment:
            start = i
            in_segment = True
        elif not is_eval[i] and in_segment:
            segments.append((start, i - 1))
            in_segment = False
    if in_segment:
        segments.append((start, n - 1))
        
    for start_idx, end_idx in segments:
        L = start_idx - 1
        R = end_idx + 1
        
        has_L = L >= 0
        has_R = R < n
        
        raw_res = np.zeros(n)
        for i in range(start_idx, end_idx + 1):
            prev_val = raw_res[i - 1] if i > start_idx else 0.0
            raw_res[i] = prev_val + pred_deltas[i]
            
        if has_L and has_R:
            val_at_R = raw_res[end_idx] + pred_deltas[R]
            closure_error = 0.0 - val_at_R
            
            segment_length = R - L
            for i in range(start_idx, end_idx + 1):
                fraction = (i - L) / segment_length
                pred_residuals[i] = raw_res[i] + (closure_error * fraction)
                
        elif has_L:
            for i in range(start_idx, end_idx + 1):
                pred_residuals[i] = raw_res[i]
                
        elif has_R:
            raw_res_back = np.zeros(n)
            for i in range(end_idx, start_idx - 1, -1):
                next_val = raw_res_back[i + 1] if i < end_idx else 0.0
                raw_res_back[i] = next_val - pred_deltas[i + 1]
            for i in range(start_idx, end_idx + 1):
                pred_residuals[i] = raw_res_back[i]
                
        else:
            for i in range(start_idx, end_idx + 1):
                pred_residuals[i] = raw_res[i]
                
    well_df['Pred_Residual'] = pred_residuals
    well_df['Pred_C'] = well_df['Pred_Residual'] + well_df['Interpolated_C']
    well_df['Pred_TVT'] = well_df['Pred_C'] - well_df['Z']
    
    return well_df

def main():
    input_paths = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path("./rogii-wellbore-geology-prediction")
    ]
    
    comp_dir = None
    for path in input_paths:
        if path.exists():
            comp_dir = path
            break
            
    if comp_dir is None:
        raise FileNotFoundError("Could not locate the competition dataset directory. Check input paths.")
        
    train_dir = comp_dir / "train"
    test_dir = comp_dir / "test"
    sample_sub_path = comp_dir / "sample_submission.csv"
    
    print(f"Dataset root found: {comp_dir}")
    print("Loading training data and computing advanced features...")
    df_full = load_all_delta_wells(train_dir, is_test=False)

    unique_wells = df_full['WELLNAME'].unique()
    train_wells, val_wells = train_test_split(unique_wells, test_size=0.20, random_state=42)
    
    train_df = df_full[df_full['WELLNAME'].isin(train_wells)].copy()
    val_df = df_full[df_full['WELLNAME'].isin(val_wells)].copy()
    
    scaler = StandardScaler()
    train_df[['X_scaled', 'Y_scaled']] = scaler.fit_transform(train_df[['X', 'Y']])
    val_df[['X_scaled', 'Y_scaled']] = scaler.transform(val_df[['X', 'Y']])
    
    features = [
        'Z', 'X_scaled', 'Y_scaled', 'MD', 'relative_md',
        'GR_clean', 'GR_roll_10', 'GR_roll_50', 
        'dGR', 'dGR_roll_10', 'dGR_roll_50',
        'Interpolated_C', 'dZ_dMD', 'dZ', 'dMD',
        'dX', 'dY', 'azimuth',
        'Typewell_GR_at_Est_TVT', 'GR_to_Typewell_Ratio', 'GR_to_Typewell_Diff',
        'typewell_alignment_score',
        'well_known_mean_C', 'well_known_std_C'
    ]
    
    X_train = train_df[features]
    y_train = train_df['Target_Delta_Residual']
    
    X_val = val_df[features]
    y_val = val_df['Target_Delta_Residual']
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.02,
        'num_leaves': 127,
        'max_depth': 9,
        'min_child_samples': 50,
        'feature_fraction': 0.7,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'reg_alpha': 1.0,
        'reg_lambda': 5.0,
        'verbose': -1,
        'random_state': 42
    }
    
    train_dataset = lgb.Dataset(X_train, label=y_train)
    val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)
    
    model = lgb.train(
        params,
        train_dataset,
        num_boost_round=1500,
        valid_sets=[val_dataset],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
    )
    
    val_df['Pred_Delta_Residual'] = model.predict(X_val)
    
    print("\nEvaluating validation set reconstruction using Brownian Bridge...")
    reconstructed_wells = []
    for _, group in val_df.groupby('WELLNAME'):
        reconstructed_well = reconstruct_well_using_bridge(group, group['Pred_Delta_Residual'].values)
        reconstructed_wells.append(reconstructed_well)
        
    val_df_reconstructed = pd.concat(reconstructed_wells, ignore_index=True)
    val_eval_subset = val_df_reconstructed[val_df_reconstructed['is_eval_zone'] == 1]
    rmse_tvt = root_mean_squared_error(val_eval_subset['TVT'], val_eval_subset['Pred_TVT'])
    
    print("="*50)
    print(f"Validation TVT RMSE (Eval Zone Only): {rmse_tvt:.4f} ft")
    print("="*50)
    
    best_rounds = model.best_iteration if model.best_iteration > 0 else 600
    
    scaler_full = StandardScaler()
    df_full[['X_scaled', 'Y_scaled']] = scaler_full.fit_transform(df_full[['X', 'Y']])
    
    X_full = df_full[features]
    y_full = df_full['Target_Delta_Residual']
    
    full_dataset = lgb.Dataset(X_full, label=y_full)
    final_model = lgb.train(
        params,
        full_dataset,
        num_boost_round=best_rounds
    )
    
    test_df = load_all_delta_wells(test_dir, is_test=True)
    test_df[['X_scaled', 'Y_scaled']] = scaler_full.transform(test_df[['X', 'Y']])
    test_df['Pred_Delta_Residual'] = final_model.predict(test_df[features])
    
    reconstructed_test = []
    for _, group in test_df.groupby('WELLNAME'):
        recon_test_well = reconstruct_well_using_bridge(group, group['Pred_Delta_Residual'].values)
        reconstructed_test.append(recon_test_well)
        
    test_df_reconstructed = pd.concat(reconstructed_test, ignore_index=True)
    test_df_reconstructed['id'] = test_df_reconstructed['WELLNAME'] + '_' + test_df_reconstructed['row_index'].astype(str)
    
    sub_df = pd.read_csv(sample_sub_path)
    
    id_to_tvt_map = dict(zip(test_df_reconstructed['id'], test_df_reconstructed['Pred_TVT']))
    sub_df['tvt'] = sub_df['id'].map(id_to_tvt_map)
    
    # Clean up missing records or potential NaNs safely with the mean TVT
    fallback_value = df_full['TVT'].mean() if 'TVT' in df_full.columns else 20.0
    sub_df['tvt'] = sub_df['tvt'].fillna(fallback_value)
    
    # Save target output
    output_path = "submission.csv"
    sub_df.to_csv(output_path, index=False)
    print(f"Saved successfully: {output_path} ({len(sub_df)} rows predicted)")

if __name__ == "__main__":
    main()

Dataset root found: /kaggle/input/competitions/rogii-wellbore-geology-prediction
Loading training data and computing advanced features...
[100]	valid_0's rmse: 0.0290086
[200]	valid_0's rmse: 0.0285084
[300]	valid_0's rmse: 0.0284527
[400]	valid_0's rmse: 0.0284368
[500]	valid_0's rmse: 0.0284171
[600]	valid_0's rmse: 0.0284111

Evaluating validation set reconstruction using Brownian Bridge...
Validation TVT RMSE (Eval Zone Only): 15.5265 ft
Saved successfully: submission.csv (14151 rows predicted)
